# 18 · Tools and `ToolNode` (no LLM yet)

**Goal:** by the end you can turn Python functions into **tools**, run them the way an LLM agent does with `ToolNode`
and `tools_condition`, and read the messages that pass between them. All of it runs with no LLM and no API key.

## The idea in plain English

An LLM can't do arithmetic reliably, but it can ask a program to do it. A **tool** is a Python function wrapped with a
name, a description and a list of arguments, so that an LLM can read what it does and ask to use it.

Here is the whole conversation an LLM agent has with its tools:

1. The LLM replies with an `AIMessage` holding **tool calls**: "please run `multiply` with `a=12.5, b=8`". Each tool
   call has an `id`.
2. **`ToolNode`** runs each requested tool and replies with a **`ToolMessage`** per call. Each one carries the result
   and the matching **`tool_call_id`**, so everyone knows which answer goes with which request.
3. **`tools_condition`**, a ready-made router, looks at the last message. If it asks for tools, it goes to the tools.
   If not, the agent is finished.

In this lesson **we** write the `AIMessage` by hand, playing the LLM. That shows all of the machinery with zero API
calls. In lesson 19, Gemini writes those messages instead.

**Analogy:** a manager (the LLM) writes job tickets, each with a ticket number. The workshop (`ToolNode`) does each
job and staples the result to the ticket with the same number.

## Picture

The agent loop. `model` is a fake LLM in this lesson and a real one in lesson 19. Nothing else changes.

```mermaid
flowchart LR
    S([START]) --> model
    model -. "tools_condition: has tool calls" .-> tools
    model -. "tools_condition: no tool calls" .-> E([END])
    tools --> model
```

## Part 1: Tools

### Step 1: Make tools with `@tool`

The **`@tool`** decorator (from `langchain.tools`) turns a function into a tool. A **decorator** is the `@name` line
above a function. It wraps the function in something extra. Two things matter, because the LLM reads them:

- the **type hints** (`a: float`) become the argument list,
- the **docstring** becomes the description.

In [1]:
from langchain.tools import tool


@tool
def add(a: float, b: float) -> float:
    """Add two numbers."""
    return a + b


@tool
def subtract(a: float, b: float) -> float:
    """Subtract b from a."""
    return a - b

Two more tools, written the same way. We collect all four in a list called `tools`.

In [2]:
@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b


@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


tools = [add, subtract, multiply, divide]

### Step 2: What the LLM will see

A tool has a `name`, a `description` and `args`, all taken from the function. `tool_call_schema` shows the full
description in JSON Schema form (lesson 11), which is roughly what's sent to the LLM.

In [3]:
print("name:       ", add.name)
print("description:", add.description)
print("args:       ", add.args)
print()
print(add.tool_call_schema.model_json_schema())

name:        add
description: Add two numbers.
args:        {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}

{'description': 'Add two numbers.', 'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'add', 'type': 'object'}


### Step 3: Run a tool directly

You run a tool with `.invoke` and a dict of arguments. The arguments are validated with Pydantic (lesson 02), so the
text `"2"` becomes the number `2.0`.

In [4]:
print(add.invoke({"a": 2, "b": 3}))
print(add.invoke({"a": "2", "b": 3}))

5.0
5.0


## Part 2: Tool calls and `ToolNode`

### Step 4: Play the LLM, a hand-made `AIMessage`

This is exactly the kind of message an LLM sends when it wants tools to run. `content` is empty (it has nothing to
say yet), and `tool_calls` lists two requests, each with a `name`, `args` and a unique `id`.

In [5]:
from langchain.messages import AIMessage

fake_ai = AIMessage(content="", tool_calls=[
    {"name": "add", "args": {"a": 2, "b": 3}, "id": "call_1", "type": "tool_call"},
    {"name": "multiply", "args": {"a": 4, "b": 5}, "id": "call_2", "type": "tool_call"},
])
for call in fake_ai.tool_calls:
    print(call)

{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_1', 'type': 'tool_call'}
{'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_2', 'type': 'tool_call'}


### Step 5: Run the tool calls with `ToolNode`

**`ToolNode`** (from `langgraph.prebuilt`) is a ready-made node that runs the tool calls in the last message. It
works on a `messages` key, so we use `MessagesState` (lesson 06).

In the installed LangGraph version, a `ToolNode` must run **inside a graph**. Calling `ToolNode(tools).invoke(...)` on
its own fails (`ValueError: Missing required config key ...`). So we put it in a one-node graph:
`START → tools → END`.

In [6]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode

builder = StateGraph(MessagesState)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
tool_runner = builder.compile()

out = tool_runner.invoke({"messages": [fake_ai]})
print(out)

{'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={}, id='698bcc4f-0333-44dd-8a88-258171033c4b', tool_calls=[{'name': 'add', 'args': {'a': 2, 'b': 3}, 'id': 'call_1', 'type': 'tool_call'}, {'name': 'multiply', 'args': {'a': 4, 'b': 5}, 'id': 'call_2', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content='5.0', name='add', id='b129d11e-a326-4ebe-98fb-7c1f49ca392e', tool_call_id='call_1'), ToolMessage(content='20.0', name='multiply', id='74aaad49-de5a-4226-ab7f-8e83f29d3218', tool_call_id='call_2')]}


### Step 6: Read the `ToolMessage`s

`ToolNode` appended one **`ToolMessage`** per tool call. Its `tool_call_id` matches the `id` of the request it
answers. That's how an LLM pairs up answers with questions when it asked for several at once.

In [7]:
for message in out["messages"][1:]:  # [0] is our fake AIMessage
    print(f"{type(message).__name__}: name={message.name}, tool_call_id={message.tool_call_id}, "
          f"content={message.content!r}")

ToolMessage: name=add, tool_call_id=call_1, content='5.0'
ToolMessage: name=multiply, tool_call_id=call_2, content='20.0'


### Step 7: `tools_condition`, the ready-made router

**`tools_condition`** reads the last message. If it has tool calls it returns `"tools"`; otherwise it returns `END`
(`'__end__'`). It's a router like the ones in lesson 05, written for you.

In [8]:
from langgraph.prebuilt import tools_condition

print(tools_condition({"messages": [fake_ai]}))
print(tools_condition({"messages": [AIMessage(content="The answer is 5.")]}))

tools
__end__


## Part 3: The whole agent loop, with a fake LLM

### Step 8: A scripted "model" node

`fake_model` pretends to be an LLM, following a fixed script:

- if the last message is a question, reply with a tool call to `multiply`,
- if the last message is a tool's result, reply with a final answer (no tool calls).

In [9]:
from langchain.messages import HumanMessage, ToolMessage


def fake_model(state: MessagesState) -> dict:
    last = state["messages"][-1]
    if isinstance(last, ToolMessage):
        return {"messages": [AIMessage(content=f"12.5 times 8 is {last.content}.")]}
    tool_call = {"name": "multiply", "args": {"a": 12.5, "b": 8}, "id": "call_1", "type": "tool_call"}
    return {"messages": [AIMessage(content="", tool_calls=[tool_call])]}

### Step 9: Wire the loop

`model` → `tools_condition` → either `tools` or `END`. After `tools`, always back to `model`, so it can read the
result. That's the loop from lesson 07, with the ready-made router deciding when to stop.

In [10]:
builder = StateGraph(MessagesState)
builder.add_node("model", fake_model)
builder.add_node("tools", ToolNode(tools))
builder.add_edge(START, "model")
builder.add_conditional_edges("model", tools_condition)
builder.add_edge("tools", "model")

agent = builder.compile()
print(agent.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	model(model)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> model;
	model -.-> __end__;
	model -.-> tools;
	tools --> model;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Step 10: Run it and read the conversation

`pretty_print()` prints a message in a readable box. Read it top to bottom: the question, the model asking for a tool,
the tool's answer, and the model's final reply.

In [11]:
final = agent.invoke({"messages": [HumanMessage("What is 12.5 times 8?")]})
for message in final["messages"]:
    message.pretty_print()

================================ Human Message =================================

What is 12.5 times 8?
================================== Ai Message ==================================
Tool Calls:
  multiply (call_1)
 Call ID: call_1
  Args:
    a: 12.5
    b: 8
================================= Tool Message =================================
Name: multiply

100.0
================================== Ai Message ==================================

12.5 times 8 is 100.0.


## Part 4: When a tool fails

### Step 11: `handle_tool_errors=True`

If a tool raises an error (dividing by zero), the default `ToolNode` lets the error crash the run. With
`handle_tool_errors=True`, it catches the error and sends it back as a `ToolMessage` with `status='error'` instead. An
LLM can then read the error and try again.

In [12]:
zero = AIMessage(content="", tool_calls=[
    {"name": "divide", "args": {"a": 1, "b": 0}, "id": "call_z", "type": "tool_call"},
])

builder = StateGraph(MessagesState)
builder.add_node("tools", ToolNode(tools, handle_tool_errors=True))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
safe_runner = builder.compile()

reply = safe_runner.invoke({"messages": [zero]})["messages"][-1]
print("status: ", reply.status)
print("content:", reply.content)

status:  error
content: Error: ZeroDivisionError('float division by zero')
 Please fix your mistakes.


Some mistakes are turned into error messages even **without** that option: asking for a tool that doesn't exist, or
leaving out an argument.

In [13]:
unknown = AIMessage(content="", tool_calls=[
    {"name": "power", "args": {"a": 2, "b": 3}, "id": "call_p", "type": "tool_call"},
])
reply = tool_runner.invoke({"messages": [unknown]})["messages"][-1]
print("status: ", reply.status)
print("content:", reply.content)

status:  error
content: Error: power is not a valid tool, try one of [add, subtract, multiply, divide].


## What just happened

- **Step 2** printed what an LLM sees about `add`: the name `add`, the description `Add two numbers.` and the args `a`
  and `b`, both numbers.
- **Step 3** ran `add` directly. `{"a": "2", "b": 3}` was converted and gave `5.0`.
- **Steps 4–6** ran two hand-written tool calls through `ToolNode`. It returned two `ToolMessage`s, `5.0` for
  `call_1` and `20.0` for `call_2`, matched by `tool_call_id`.
- **Step 7** showed `tools_condition` returning `'tools'` for a message with tool calls and `'__end__'` without.
- **Steps 8–10** ran the full agent loop with a fake model: Human → AI (tool call `multiply`) → Tool (`100.0`) → AI
  (`12.5 times 8 is 100.0.`).
- **Step 11** turned a division by zero into a `ToolMessage` with `status: error`, and showed an unknown tool name
  `power` being reported the same way.

## Common mistakes

1. **A tool function without a docstring.** The LLM needs a description, so `@tool` refuses:
   `ValueError: Function must have a docstring if description not provided.`
   Fix: add a one-line docstring saying what the tool does.
2. **Calling a tool like a normal function.** After `@tool`, `add(2, 3)` no longer works:
   `TypeError: 'StructuredTool' object is not callable`.
   Fix: `add.invoke({"a": 2, "b": 3})`.
3. **A tool that raises, with the default `ToolNode`.** `divide` with `b=0` crashes the whole run:
   `ZeroDivisionError: float division by zero`.
   Fix: `ToolNode(tools, handle_tool_errors=True)`, so the error goes back to the model as a message.

## Try it

1. Add a `power(a, b)` tool (`a ** b`) to `tools`, rebuild `tool_runner`, and send it a hand-written tool call for
   `2 ** 10`. What does the `ToolMessage` contain?
2. Change `fake_model` so its first reply asks for **two** tools at once: `add(3, 5)` and `multiply(3, 5)`. Its final
   reply should mention both results. (Hint: when the last message is a `ToolMessage`, the two results are the last two
   messages.)

Answers are in [`solutions/exercises_solutions.ipynb`](../solutions/exercises_solutions.ipynb).

## Key terms

New terms from this lesson, also in [GLOSSARY.md](../GLOSSARY.md):

- **Tool**: a function an LLM can ask to run, with a name, a description and typed arguments.
- **`@tool`**: the decorator that turns a function into a tool. Its docstring and type hints become the description
  and arguments.
- **Decorator**: an `@name` line above a function that wraps it in extra behaviour.
- **Tool call**: a request inside an `AIMessage` to run one tool, with a `name`, `args` and `id`.
- **`ToolMessage`**: the reply to a tool call, holding the result and the matching `tool_call_id`.
- **`tool_call_id`**: the ID that links a `ToolMessage` to the tool call it answers.
- **`ToolNode`**: a ready-made node that runs the tool calls in the last message.
- **`tools_condition`**: a ready-made router: `"tools"` if the last message has tool calls, otherwise `END`.
- **Agent loop**: model → tools → model … until the model answers without tool calls.